# What data does each alignment method need, and what does it change in a 0.5B model?

Question: a base model only continues text. Which training method turns it into one that follows instructions? What data does each method need, what does each one change, and what does each one break?

Setup: two Tesla T4 GPUs (15 GiB each, Turing, no bf16). Base model `Qwen/Qwen2.5-0.5B`. Instruct model `Qwen/Qwen2.5-0.5B-Instruct`. Teacher `Qwen/Qwen2.5-1.5B-Instruct`. TRL 1.18.2 in a `uv venv` with pinned versions. Training uses the full model in fp32 with LoRA adapters, so fp16 overflow and NaN cannot occur. Each stage logs a finite-loss check. Run time: about 70 minutes. The notebook starts no stage after minute 84. It needs no API key.

End token. Every reply in training ends with `<|endoftext|>`. The `<|im_end|>` row of the base model may be untrained, and LoRA does not train embeddings. The prep stage prints the evidence: norm and cosine of that row against 50 unused rows. Generation stops on either token. The prompt template is the Qwen chat template.

Data. All training and test items come from code with fixed seeds. There are 6 instruction families. Each has a grader that code runs. Sizes: 1,500 train items, 40 dev items, 120 held-out items, no prompt in two sets (an assert checks this). Arithmetic for GRPO: 3,000 train items and 100 eval items from code. Forgetting probe: 50 test items of GSM8K (`openai/gsm8k`, MIT licence).

Objectives:

- `customization.3`: SFT, PEFT, LoRA, QLoRA, RLHF, DPO, KTO, GRPO, distillation, instruction tuning. Measure what data each needs and what each changes.
- `models.7`: base against instruction-tuned, small against larger model, foundation against task-specific.

Terms used below. Sources are in `hub/sources/`. A source file is named where it supports the line. A term with no source file is defined by what this notebook measures.

- **Base model**, **instruction-tuned model**: no source file defines them. Part A measures the difference between the two checkpoints.
- **SFT** (supervised fine-tuning): train on prompt and right-reply pairs. The loss is the next-token error on the reply (`mastering-agentic-techniques-ai-agent-customization.txt`: SFT imitates good examples).
- **PEFT, LoRA**: freeze most weights and train small added matrices (same source). **QLoRA**: LoRA on a base model with quantized linear layers (`qlora.txt`).
- **Preference pair**: one prompt, a chosen reply, a rejected reply. **DPO** trains on pairs and needs no separate reward model (same agentic source). **KTO** needs only single replies with a good or bad mark (`gtc24-s62092.txt`).
- **RLHF**: reinforcement learning from human feedback. It uses a separate reward model (agentic source) and adapts behaviour from feedback (`selecting-large-language-model-customization-techniques.txt`). **Best-of-n**: sample n replies and keep the one the reward model likes most. **Reward hacking**: the reward rises and the true score does not.
- **GRPO**: sample a group of replies per prompt and score each with a reward function. No critic network. Fits tasks with a checkable answer (agentic source).
- **Distillation**: train a small model from a larger one (`build-efficient-ai-agents-through-model-distillation-with-nvidias-data-flywheel-blueprint.txt`). *Sequence level*: train on the teacher's replies. *Logit level*: match the teacher's token probabilities.
- **Cut reply**: a reply that never stopped inside the token limit. It never counts as right.
- **Forgetting**: the loss on a task the training did not cover (here GSM8K).
- **Wilson interval**: 95% interval for a rate. **Exact McNemar test**: paired test for two systems on the same items.

## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Step | What to guess | Your guess |
|---|---|---|
| A. Base against instruct | Share of 40 instructions followed by each model. How many base replies are cut? | |
| A. Thinking tokens | Does room to think change arithmetic accuracy? | |
| B. SFT | Held-out share followed after SFT. Compare with the instruct model. | |
| B. QLoRA | Peak GPU memory and quality against LoRA on the same data | |
| C. DPO against KTO | Which one changes reply length more? Does either beat SFT by more than noise? | |
| D. Reward model | Accuracy on the pair type it never saw (right-long over wrong-short) | |
| D. Best-of-16 | Does the true score rise as fast as the reward? | |
| E. GRPO | Mean reward at the start and at the end. Arithmetic accuracy after training. | |
| F. Distillation | Teacher replies against code replies. Do logits add anything? | |
| G. Forgetting | Which method loses most on GSM8K? | |

## Setup

Cells 1 to 7 build the tools. Cell 2 writes one shared file with the data generators, the graders and the statistics. The stage scripts import the same file, so the notebook and the stages agree on every item. Cell 3 tests every grader on right, wrong, empty and tricky replies. It then runs each test against a broken grader, and the test must fail there. Cell 4 builds the training environment. Cells 5 and 6 run all training stages.

The kernel does not import torch or TRL. Heavy work runs in subprocesses, so a crash cannot stop the notebook.

In [ ]:
# 1. Helpers: the part runner, a timer, and a table printer. This kernel never imports torch.
import ast, json, os, re, shutil, signal, subprocess, sys, threading, time, traceback
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

T0 = time.time(); DEADLINE = T0 + 84 * 60            # every stage obeys this wall-clock limit, so the notebook ends in under 90 minutes
OUT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("out"); OUT.mkdir(exist_ok=True)
PARTS = {}
def lap(msg): print(f"[{(time.time() - T0) / 60:5.1f} min] {msg}", flush=True)

def part(name, fn):  # run one part. A failure prints its traceback and the notebook goes on
    t = time.time()
    try: fn(); PARTS[name] = ("ok", time.time() - t)
    except Exception as e: traceback.print_exc(); PARTS[name] = (f"FAILED {type(e).__name__}: {str(e)[:70]}", time.time() - t)
    lap(f"part {name}: {PARTS[name][0]}")

def show(name, df):  # print the whole table to the log (stdout only) and save it as CSV
    df.to_csv(OUT / f"{name}.csv", index=False)
    f = lambda v: v[:40] if isinstance(v, str) else (round(v, 3) if isinstance(v, float) else v)
    print(f"--- {name}"); print("\n".join(l[:199] for l in df.apply(lambda c: c.map(f)).to_string(index=False).splitlines()))
lap("helpers ready")

In [ ]:
# 2. The shared code file: data generators, graders, statistics, model helpers. The stage scripts import this same file.
W = Path(os.environ.get("ALIGN_DIR", "/kaggle/working/align")); (W / "stages").mkdir(parents=True, exist_ok=True); (W / "logs").mkdir(exist_ok=True)
COMMON_SRC = r'''
# common.py: data, graders, statistics and model helpers. The notebook and every stage script import this file.
import hashlib, json, math, os, random, re, time
from pathlib import Path

W = Path(os.environ.get("ALIGN_DIR", "/kaggle/working/align"))                      # code, logs, result JSON
M = Path(os.environ.get("ALIGN_MODELS", "/kaggle/temp/align_models" if Path("/kaggle/temp").is_dir() else str(W / "models")))  # venv, weights
IM_END, EOT = "<|im_end|>", "<|endoftext|>"
THINGS = "moon sun river bread chair train clock garden phone bridge cloud mirror candle ladder window rocket pencil tunnel forest harbor".split()
ADJ = "bright quiet old fast soft tall cold warm heavy small sharp green".split()
CATS = {k: v.split() for k, v in {
    "fruits": "apple pear plum grape lemon mango peach cherry", "animals": "cat dog horse sheep tiger zebra whale eagle",
    "colors": "red blue green pink black white brown gray", "tools": "hammer saw drill wrench pliers chisel shovel rake",
    "cities": "paris rome oslo lima cairo delhi tokyo dublin", "sports": "tennis rugby golf chess boxing rowing soccer hockey",
    "metals": "iron gold zinc lead tin copper silver nickel", "planets": "mars venus earth saturn jupiter mercury uranus neptune"}.items()}
KEYS = "name city color pet food team tool song book plant".split()
VALS = "Anna Ben Cleo Dev Eli Fay Gus Hana Ivo Jun Kai Lena Milo Nora Omar Pia Quin Rita Sam Tara Uma Vik Wen Xia Yara Zed Alma Bruno Chloe Dario".split()
FAMS = ["arith", "json", "listn", "maxwords", "lower", "prefix"]

# ---------- data: every item has a spec, a grader (check), and replies that code writes ----------
def pool(fam):
    if fam == "arith": return [(a, b, o) for a in range(10, 100) for b in range(10, 100) for o in "+-" if o == "+" or a > b]
    if fam == "expr": return [(a, b, c) for a in range(2, 10) for b in range(11, 40) for c in range(11, 100)]
    if fam == "json": return [(k, v) for k in KEYS for v in VALS]
    if fam == "listn": return [(c, n, al) for c in CATS for n in range(2, 7) for al in (0, 1)]
    if fam == "maxwords": return [(t, n, s) for t in THINGS for n in (8, 9, 10) for s in (0, 1)]
    if fam == "lower": return [(t, w) for t in THINGS for w in ADJ]
    return [(t, p) for t in THINGS for p in ("Note", "Tip", "Hint", "Fact")]

def spec(fam, p):
    if fam == "arith":
        a, b, o = p
        return dict(fam=fam, a=a, b=b, o=o, g=a + b if o == "+" else a - b, user=f"What is {a} {o} {b}? Reply with a last line in the form Answer: <number>")
    if fam == "expr":
        a, b, c = p
        return dict(fam=fam, a=a, b=b, c=c, g=a * b + c, user=f"Compute {a} * {b} + {c}. Think step by step, then finish with a last line in the form Answer: <number>")
    if fam == "json":
        k, v = p
        return dict(fam=fam, k=k, v=v, user=f'Reply with only a JSON object with the key "{k}" set to the string "{v}".')
    if fam == "listn":
        c, n, al = p
        return dict(fam=fam, c=c, n=n, al=al, user=f"List exactly {n} {c}, one per line, with no other text." + (" Put them in alphabetical order." if al else ""))
    if fam == "maxwords":
        t, n, s = p
        return dict(fam=fam, t=t, n=n, user=f"Describe the {t} in at most {n} words." if s == 0 else f"In at most {n} words, describe the {t}.")
    if fam == "lower":
        t, w = p
        return dict(fam=fam, t=t, w=w, user=f'Write one sentence about the {t} that uses the word "{w}", in all lowercase letters.')
    t, pf = p
    return dict(fam=fam, t=t, pf=pf, user=f'Start your reply with "{pf}:" and then give one short sentence about the {t}.')

def direct(s):  # same expression, but no room to think
    return s["user"].replace("Think step by step, then finish with a last line", "Reply with only one line")

def last_answer(reply):  # only the LAST non-empty line counts, and it must be exactly "Answer: <integer>"
    ls = [l.strip() for l in reply.strip().splitlines() if l.strip()]
    m = re.fullmatch(r"Answer:\s*(-?\d+)", ls[-1]) if ls else None
    return int(m.group(1)) if m else None

def check(s, reply):
    f, t = s["fam"], reply.strip()
    if f in ("arith", "expr"): return last_answer(reply) == s["g"]
    if f == "json":
        try: return json.loads(t) == {s["k"]: s["v"]}
        except ValueError: return False
    if f == "listn":
        ls = t.splitlines()
        return (len(ls) == s["n"] and all(re.fullmatch(r"[a-z]+( [a-z]+)*", l) and l.split()[-1] in CATS[s["c"]] for l in ls)
                and len(set(ls)) == s["n"] and (not s["al"] or ls == sorted(ls)))
    ws = [x.strip(".,!?:;\"").lower() for x in t.split()]
    one = "\n" not in t
    if f == "maxwords": return one and 3 <= len(ws) <= s["n"] and s["t"] in ws
    if f == "lower": return one and t == t.lower() and t.endswith(".") and len(ws) >= 4 and s["t"] in ws and s["w"] in ws
    return one and t.startswith(s["pf"] + ":") and 4 <= len(ws) <= 25 and s["t"] in ws       # prefix

def reply(s, rng, kind):  # kind: gold_s gold_l bad_s bad_l = (right or wrong) x (short or long)
    f, good, long = s["fam"], kind.startswith("gold"), kind.endswith("l")
    a1, a2, a3 = rng.sample(ADJ, 3)
    if f in ("arith", "expr"):
        x = s["g"] if good else s["g"] + rng.choice([-10, -1, 1, 10])
        work = f"{s['a']} {s['o']} {s['b']} = {x}." if f == "arith" else f"{s['a']} * {s['b']} = {s['a'] * s['b']}. {s['a'] * s['b']} + {s['c']} = {x}."
        return f"{work}\nAnswer: {x}" if long else f"Answer: {x}"
    if f == "json":
        d = {s["k"]: s["v"] if good else rng.choice([v for v in VALS if v != s["v"]])}
        if good: return json.dumps(d, indent=2) if long else json.dumps(d)
        return "Here is the JSON you asked for:\n" + json.dumps(d) if long else json.dumps(d)
    if f == "listn":
        its = rng.sample(CATS[s["c"]], s["n"] if good else s["n"] + (1 if long else -1))
        ls = [f"{rng.choice(ADJ)} {i}" if long else i for i in its]
        return "\n".join(sorted(ls) if s["al"] else ls)
    t = s["t"]
    if f == "maxwords":
        if good: return f"The {t} is {a1} and {a2} and {a3}" if long else f"The {t} is {a1}."
        return " ".join(["The", t, "is", a1, "and", a2, "and", a3, "and", "very", "truly", a1][: s["n"] + 2]) if long else f"It is {a1}."
    if f == "lower":
        txt = f"the {t} is {s['w']}" + (" and i see it every day near my house" if long else "") + "."
        return txt if good else txt[0].upper() + txt[1:]
    body = f"the {t} is {a1}" + (" and people often notice it first" if long else "") + "."
    return f"{s['pf']}: {body}" if good else (f"Remember: {body}" if long else body)

def overlap(a, b): return {s["user"] for s in a} & {s["user"] for s in b}   # prompts that two item lists share

def splits(seed=0):  # train / dev (40) / held (120): the prompts never overlap
    rng, out = random.Random(seed), {"train": [], "dev": [], "held": []}
    for i, fam in enumerate(FAMS):
        P = pool(fam); rng.shuffle(P); nd = 7 if i < 4 else 6
        out["dev"] += [spec(fam, p) for p in P[:nd]]
        out["held"] += [spec(fam, p) for p in P[nd:nd + 20]]
        out["train"] += [spec(fam, rng.choice(P[nd + 20:])) for _ in range(250)]
    for v in out.values(): rng.shuffle(v)
    assert not (overlap(out["train"], out["dev"]) | overlap(out["train"], out["held"]) | overlap(out["dev"], out["held"])), "prompt in two splits"
    assert len(out["dev"]) == 40 and len(out["held"]) == 120
    return out

def expr_split(seed=1):  # arithmetic items for GRPO: 3,000 train, 100 eval, no overlap
    P = pool("expr"); random.Random(seed).shuffle(P)
    out = {"eval": [spec("expr", p) for p in P[:100]], "train": [spec("expr", p) for p in P[100:3100]]}
    assert not overlap(out["eval"], out["train"]), "prompt in train and eval"
    return out

def expr_reward(rep, gold):  # GRPO reward: 1 for the right last line, 0.1 for a well-formed wrong one, 0 otherwise
    a = last_answer(rep)
    return 0.0 if a is None else (1.0 if a == gold else 0.1)

def prefer(s, a, b):  # the exact preference rule: a reply that passes beats one that fails. Both pass: the shorter wins. Both fail: tie. +1 a wins, -1 b wins, 0 tie
    ka, kb = (check(s, a), -len(a) * check(s, a)), (check(s, b), -len(b) * check(s, b))
    return (ka > kb) - (ka < kb)

KINDS = {"K1": ("gold_s", "gold_l"), "K2": ("gold_s", "bad_l"), "K3": ("gold_l", "bad_s")}   # K1 right-short over right-long, K2 right-short over wrong-long, K3 right-long over wrong-short
def pairs(items, seed, n, kinds=("K1", "K2")):  # training pairs. With K1 and K2 only, the chosen reply is always right and always shorter (a planted length confound)
    rng, out = random.Random(seed), []
    for s in items[:n]:
        k = rng.choice(kinds); c, r = reply(s, rng, KINDS[k][0]), reply(s, rng, KINDS[k][1])
        assert check(s, c) and prefer(s, c, r) == 1 and (k == "K3" or len(c) < len(r)) and (check(s, r) == (k == "K1"))
        out.append(dict(s=s, prompt=s["user"], chosen=c, rejected=r, kind=k))
    return out

def held_pairs(items, seed=5):  # test pairs: K3 is right-long over wrong-short, where length points the wrong way
    rng, out = random.Random(seed), []
    for s in items:
        for k, (a, b) in KINDS.items():
            c, r = reply(s, rng, a), reply(s, rng, b)
            assert prefer(s, c, r) == 1 and (k != "K3" or len(c) > len(r))
            out.append(dict(s=s, prompt=s["user"], chosen=c, rejected=r, kind=k))
    return out

def sft_rows(items, seed):  # (prompt, reply) with a right reply of random length
    rng = random.Random(seed)
    return [dict(user=s["user"], completion=reply(s, rng, rng.choice(["gold_s", "gold_l"]))) for s in items]

def kto_rows(ps, seed):  # unpaired: one reply with a good or bad mark
    rows = [dict(user=p["prompt"], completion=p[k], label=k == "chosen") for p in ps for k in ("chosen", "rejected")]
    random.Random(seed).shuffle(rows); return rows

PROBES = {"empty": "", "OK": "OK", "Sure!": "Sure!", "Answer: 0": "Answer: 0", "yes x30": "yes " * 30}   # junk replies for the reward-hacking check

def data_hash():
    S, E = splits(), expr_split()
    return hashlib.sha1("\n".join(s["user"] for s in S["dev"] + S["held"] + E["eval"]).encode()).hexdigest()[:12]

# ---------- grading rows: a row is {reply, cut}. cut = the model never stopped inside the token limit ----------
def outcome(s, row):
    if row["cut"]: return "cut"
    if not row["reply"].strip(): return "empty"
    return "pass" if check(s, row["reply"]) else "wrong"

def gsm_value(row):  # first '#### n' line; else the last number if the reply ended on its own
    m = re.search(r"####\s*\$?\s*(-?[\d,]*\.?\d+)", row["reply"])
    if not m and not row["cut"]: m = re.search(r"(-?\d[\d,]*\.?\d*)\D*$", row["reply"])
    try: return float(m.group(1).replace(",", "").rstrip(".")) if m else None
    except ValueError: return None

def gsm_ok(row, gold):
    v = gsm_value(row); return v is not None and abs(v - gold) < 1e-6

# ---------- statistics ----------
def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (max(0.0, (c - h) / d), min(1.0, (c + h) / d))

def rate(k, n):
    lo, hi = wilson(k, n); return f"{k / n:.2f} [{lo:.2f},{hi:.2f}]" if n else "n/a"

def mcnemar(a_ok, b_ok):  # exact paired test on the same items. Returns (a only right, b only right, p)
    from scipy.stats import binomtest
    x = sum(a and not b for a, b in zip(a_ok, b_ok)); y = sum(b and not a for a, b in zip(a_ok, b_ok))
    return x, y, (binomtest(x, x + y, 0.5).pvalue if x + y else 1.0)

def verdict(gain, p):  # the only text conclusion we print: it comes from the numbers
    return "no reliable difference" if p >= 0.05 else ("better" if gain > 0 else "worse")

# ---------- model helpers (torch is imported inside the functions so the notebook kernel never loads it) ----------
def load_tok(path):
    from transformers import AutoTokenizer
    t = AutoTokenizer.from_pretrained(str(path)); t.padding_side = "left"; return t

def render(tok, user):
    return tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)

def eos_ids(tok): return [tok.convert_tokens_to_ids(IM_END), tok.convert_tokens_to_ids(EOT)]

def gen(model, tok, prompts, max_new=80, bs=40, n=1, temp=None):  # greedy by default. Returns (reply, cut) per prompt, n per prompt
    import torch
    ids, out = eos_ids(tok), []
    kw = dict(do_sample=True, temperature=temp, top_p=1.0, top_k=0, num_return_sequences=n) if temp else dict(do_sample=False)
    for i in range(0, len(prompts), bs):
        enc = tok(prompts[i:i + bs], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            o = model.generate(**enc, max_new_tokens=max_new, eos_token_id=ids, pad_token_id=ids[1], repetition_penalty=1.0, use_cache=True, **kw)
        for row in o[:, enc["input_ids"].shape[1]:].tolist():
            j = next((j for j, t in enumerate(row) if t in ids), None)
            out.append((tok.decode(row if j is None else row[:j], skip_special_tokens=True), j is None, "" if j is None else tok.convert_ids_to_tokens(row[j])))
    return out

def evaluate(model, tok, sets=("dev", "held", "expr", "gsm")):
    S, E = splits(), expr_split(); jobs = {"dev": (S["dev"], 80, 40), "held": (S["held"], 80, 40), "expr": (E["eval"], 160, 50)}
    jobs["expr_direct"] = ([dict(s, user=direct(s)) for s in E["eval"]], 40, 50)
    if "gsm" in sets: jobs["gsm"] = ([dict(user=r["question"] + "\nSolve it step by step. End with a final line: #### <number>") for r in json.load(open(W / "gsm8k50.json"))], 320, 25)
    return {k: [dict(reply=r, cut=c, stop=s) for r, c, s in gen(model, tok, [render(tok, s["user"]) for s in jobs[k][0]], jobs[k][1], jobs[k][2])] for k in sets}

def write(name, d):
    (W / "res").mkdir(parents=True, exist_ok=True)
    json.dump(dict(name=name, data_hash=data_hash(), **d), open(W / "res" / f"{name}.json", "w"))
    print("wrote", name, flush=True)

def finish(name, model, tok, stats, sets=("dev", "held", "expr", "gsm"), save=None, **extra):
    if save: model.save_pretrained(str(save)); tok.save_pretrained(str(save))
    write(name, dict(stats=stats, replies=evaluate(model, tok, sets), **extra))

def cfg(cls, **kw):  # one config builder: fp32 on purpose (T4 has no bf16), and keys this TRL version lacks are named, not hidden
    import dataclasses
    base = dict(bf16=False, fp16=False, report_to="none", save_strategy="no", logging_steps=2, seed=0, disable_tqdm=True,
                gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
    kw = {**base, **kw}; ok = {f.name for f in dataclasses.fields(cls)}
    if set(kw) - ok: print("config keys not in", cls.__name__, ":", sorted(set(kw) - ok), flush=True)
    return cls(**{k: v for k, v in kw.items() if k in ok})

def lora(task="CAUSAL_LM", r=16):
    from peft import LoraConfig
    return LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.0, target_modules="all-linear", task_type=task)

def run_training(build, budget_s, cap, fixed=None, probe=4, lo=10):
    # Set the step count from a measured step time: train `probe` steps on a fresh trainer, time them, then rebuild and train for real.
    import gc, torch
    from transformers import TrainerCallback
    class Tick(TrainerCallback):
        def __init__(s): s.t = []
        def on_step_end(s, args, state, control, **k): s.t.append(time.time())
    per = None
    if fixed is None:
        tr = build(probe); cb = Tick(); tr.add_callback(cb); tr.train()
        d = [b - a for a, b in zip(cb.t, cb.t[1:])]; per = sum(d) / len(d)
        del tr, cb; gc.collect(); torch.cuda.empty_cache()
    steps = int(fixed or min(cap, max(lo, budget_s // per)))
    torch.cuda.reset_peak_memory_stats(); tr = build(steps); t0 = time.time(); tr.train(); secs = time.time() - t0
    log = [{k: v for k, v in e.items() if isinstance(v, (int, float))} for e in tr.state.log_history]
    nums = [v for e in log for k, v in e.items() if k in ("loss", "grad_norm", "reward")]
    return tr, dict(steps=steps, probe_t_step=None if per is None else round(per, 2), t_step=round(secs / steps, 2), train_s=round(secs),
                    budget_s=budget_s, peak_gib=round(torch.cuda.max_memory_allocated() / 2**30, 2), nan_free=all(math.isfinite(v) for v in nums), log=log)

def merged(tr, tok):  # fold the LoRA weights into the base. Returns the plain model and the largest logit change the fold caused
    import torch
    m = tr.accelerator.unwrap_model(tr.model).eval()
    tp, tt = m.get_nb_trainable_parameters()
    enc = tok([render(tok, s["user"]) for s in splits()["dev"][:4]], return_tensors="pt", padding=True).to(m.device)
    with torch.no_grad(): a = m(**enc).logits
    mm = m.merge_and_unload().eval()
    with torch.no_grad(): b = mm(**enc).logits
    return mm, dict(trainable=tp, total=tt, merge_max_logit_diff=float((a - b).abs().max()))
'''
(W / "common.py").write_text(COMMON_SRC); sys.path.insert(0, str(W))
import common
from common import *
lap(f"common.py written to {W}")

## Data and graders (before any model runs)

Question: how do we make a preference rule that is exact?

Why the exam asks: preference methods learn from data. If the data is noisy, the method learns the noise. Here code decides every label.

The rule: a reply that passes the grader beats one that fails. When both pass, the shorter reply wins. When both fail, there is no preference. Pair types: K1 right-short over right-long, K2 right-short over wrong-long, K3 right-long over wrong-short. Training pairs are K1 and K2 only. In every training pair the chosen reply is the shorter one. This plants a length confound on purpose. K3 is a test pair type that the methods never see.

In [ ]:
# 3. Self-tests for every grader, written before any model runs. Each test also runs against a deliberately broken grader and must fail there.
import random
from common import *

def test_check(chk):
    rng = random.Random(0)
    for fam in FAMS + ["expr"]:
        for p in rng.sample(pool(fam), 25):
            s = spec(fam, p)
            for k in ("gold_s", "gold_l"): assert chk(s, reply(s, rng, k)), (fam, k)           # right replies pass
            for k in ("bad_s", "bad_l"): assert not chk(s, reply(s, rng, k)), (fam, k)         # wrong replies fail
            assert not chk(s, ""), fam                                                         # empty fails
    a = spec("arith", (40, 2, "+"))
    assert chk(a, "Answer: 42") and chk(a, "40 + 2 = 42.\nAnswer: 42\n")
    assert not chk(a, "Answer: 41") and not chk(a, "42") and not chk(a, "Answer: 42 is it")
    assert not chk(a, "Numbers: 1 2 3 40 42 4 5 6 7 8 9") and not chk(a, "Answer: 42\nand also 40 2 42 42")   # many numbers do not help
    assert not chk(a, "Answer: 42\nAnswer: 41") and chk(a, "Answer: 41\nAnswer: 42")                          # only the last line counts
    j = spec("json", ("name", "Anna"))
    assert chk(j, '{"name": "Anna"}') and not chk(j, 'Sure: {"name": "Anna"}') and not chk(j, '{"name": "Anna", "x": 1}') and not chk(j, "{")
    l = spec("listn", ("fruits", 3, 1))
    assert chk(l, "apple\npear\nplum") and not chk(l, "pear\napple\nplum") and not chk(l, "apple\npear\nplum\ngrape") and not chk(l, "1. apple\npear\nplum")
    assert not chk(l, "apple\napple\nplum") and not chk(l, "apple\n\npear\nplum") and not chk(l, "apple pear plum")
test_check(check)
try: test_check(lambda s, r: True); raise SystemExit("test_check is too weak: it accepts a grader that always says yes")
except AssertionError: pass
try: test_check(lambda s, r: False); raise SystemExit("test_check is too weak: it accepts a grader that always says no")
except AssertionError: pass

def test_reward(rw):
    assert rw("Answer: 42", 42) == 1.0 and rw("so 6*7\nAnswer: 42", 42) == 1.0
    assert rw("Answer: 41", 42) == 0.1 and rw("42", 42) == 0.0 and rw("", 42) == 0.0
    assert rw("Answer: 1 2 3 42", 42) == 0.0 and rw("Answer: 42\nok", 42) == 0.0 and rw("1 2 3 4 5 42 43 44", 42) == 0.0
test_reward(expr_reward)
try: test_reward(lambda r, g: 1.0 if str(g) in r else 0.0); raise SystemExit("test_reward is too weak")
except AssertionError: pass

def test_prefer(pf):
    a = spec("arith", (40, 2, "+")); rs, rl, bad = "Answer: 42", "40 + 2 = 42.\nAnswer: 42", "Answer: 41"
    assert pf(a, rs, rl) == 1 and pf(a, rl, rs) == -1 and pf(a, rs, bad) == 1 and pf(a, bad, rl) == -1 and pf(a, rs, rs) == 0
    assert pf(a, "", bad) == 0 and pf(a, bad, "Answer: 40") == 0 and pf(a, rs, "") == 1                # both fail: tie
test_prefer(prefer)
try: test_prefer(lambda s, a, b: 0); raise SystemExit("test_prefer is too weak")
except AssertionError: pass

def test_gsm(ok):
    r = lambda t, c=False: dict(reply=t, cut=c)
    assert ok(r("so 5+5\n#### 10"), 10) and ok(r("it is 1,000\n#### 1,000"), 1000) and ok(r("the answer is 7."), 7)
    assert not ok(r("#### 11"), 10) and not ok(r("no number"), 10) and not ok(r("12 then 10 then 99"), 10)
    assert not ok(r("it goes on 10 and", True), 10) and ok(r("#### 10 and then rambling 99", True), 10)   # a cut reply counts only with a marker
test_gsm(gsm_ok)
try: test_gsm(lambda row, g: True); raise SystemExit("test_gsm is too weak")
except AssertionError: pass

def test_stats(wil, mcn):
    lo, hi = wil(0, 10); assert lo < 1e-9 and 0.27 < hi < 0.29
    lo, hi = wil(50, 100); assert 0.40 < lo < 0.41 and 0.59 < hi < 0.60
    assert mcn([1] * 10 + [0] * 10, [0] * 10 + [1] * 10)[2] == 1.0 and mcn([1] * 10, [0] * 10)[2] < 0.01
test_stats(wilson, mcnemar)
try: test_stats(lambda k, n: (0.0, 1.0), mcnemar); raise SystemExit("test_stats is too weak")
except AssertionError: pass

S, E = splits(), expr_split()
assert splits() == S and data_hash() == data_hash()                            # fixed seeds
P, HP = pairs(S["train"], 1, 1000), held_pairs(S["held"])
assert len(P) == 1000 and len(HP) == 360 and all(prefer(p["s"], p["chosen"], p["rejected"]) == 1 for p in P + HP)   # the rule is exact on every pair
assert sum(p["kind"] == "K1" for p in P) in range(440, 560) and all(len(p["chosen"]) < len(p["rejected"]) for p in P)
assert all(len(p["chosen"]) > len(p["rejected"]) for p in HP if p["kind"] == "K3")
PB = pairs(S["train"], 2, 1000, ("K1", "K2", "K3"))                     # balanced pairs for reward model 2: the rule is still exact, and length no longer points to the chosen reply
assert all(prefer(p["s"], p["chosen"], p["rejected"]) == 1 for p in PB) and sum(len(p["chosen"]) > len(p["rejected"]) for p in PB) > 250
assert sum(p["kind"] == "K3" for p in PB) in range(280, 390) and sum(len(p["chosen"]) > len(p["rejected"]) for p in P) == 0
assert sorted(c for c in {"K1", "K2", "K3"}) == sorted({p["kind"] for p in HP})
assert not overlap(S["train"], S["held"]) and not overlap(S["train"], S["dev"]) and not overlap(S["dev"], S["held"]) and not overlap(E["train"], E["eval"])
assert overlap(S["train"], S["train"][:5]), "the overlap detector must see a planted leak"           # the detector can fail
print("train", len(S["train"]), "dev", len(S["dev"]), "held", len(S["held"]), "| expr train", len(E["train"]), "eval", len(E["eval"]), "| pairs", len(P), "held pairs", len(HP))
print("no prompt is in two of train, dev, held:", not ({s["user"] for s in S["train"]} & {s["user"] for s in S["dev"] + S["held"]}))
print("pair kinds in training:", {k: sum(p["kind"] == k for p in P) for k in ("K1", "K2")}, "| chosen is shorter in", sum(len(p["chosen"]) < len(p["rejected"]) for p in P), "of", len(P))
rng = random.Random(0)
for fam in FAMS:
    s = next(x for x in S["held"] if x["fam"] == fam)
    print(f"{fam:9s} | {s['user'][:70]!r} | gold_s {reply(s, rng, 'gold_s')[:30]!r} | bad_l {reply(s, rng, 'bad_l')[:40]!r}")
print("all grader tests passed, and each test fails on a broken grader")

## Environment

One `uv venv` holds torch 2.8.0 (CUDA 12.8 build), `trl==1.18.2`, `transformers==5.23.0`, `peft==0.22.0`, `accelerate==1.16.0`, `datasets==5.2.0`, `bitsandbytes==0.50.2`. These versions fit each other. `trl 1.18.2` needs `transformers>=4.56.2`, `accelerate>=1.11.0`, `datasets>=4.7.0` and `torch>=2.6.0`. A smoke test imports every trainer and runs one matrix product on the GPU. If it fails, the cell tries the default torch wheel.

In [ ]:
# 4. Build the training environment: one uv venv with pinned versions. The kernel never imports these packages.
PINS = ["trl==1.18.2", "transformers==5.23.0", "peft==0.22.0", "accelerate==1.16.0", "datasets==5.2.0", "bitsandbytes==0.50.2"]
SMOKE = ("import torch, trl, transformers, peft, accelerate, datasets; "
         "from trl import SFTTrainer, DPOTrainer, KTOTrainer, RewardTrainer, GRPOTrainer, DistillationTrainer; "
         "x = torch.ones(8, 8, device='cuda') @ torch.ones(8, 8, device='cuda'); assert x[0, 0] == 8; "
         "print('smoke ok | torch', torch.__version__, 'cuda', torch.version.cuda, '| trl', trl.__version__, '| transformers', transformers.__version__, "
         "'| peft', peft.__version__, '| gpus', torch.cuda.device_count(), torch.cuda.get_device_name(0))")
PY = M / "venv/bin/python"

def sh(cmd, limit):
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=limit)
    if r.returncode: print((r.stdout + r.stderr)[-1500:]); raise RuntimeError(f"{cmd[0]} exited with {r.returncode}")
    return r.stdout

def build_venv():  # first try torch 2.8.0 built for CUDA 12.8, which the T4 driver runs. Else the default torch wheel
    for label, torch_args in (("torch 2.8.0 cu128", ["torch==2.8.0", "--index-url", "https://download.pytorch.org/whl/cu128"]), ("torch from PyPI", ["torch"])):
        try:
            shutil.rmtree(M / "venv", ignore_errors=True); M.mkdir(parents=True, exist_ok=True)
            sh(["uv", "venv", str(M / "venv"), "--python", "3.13"], 300)
            sh(["uv", "pip", "install", "--python", str(PY), *torch_args], 900)
            sh(["uv", "pip", "install", "--python", str(PY), *PINS], 600)
            print(label, "->", sh([str(PY), "-c", SMOKE], 300).strip()); return
        except Exception as e: print(label, "failed:", repr(e)[:300])
    raise RuntimeError("no working venv. Every stage will be skipped")

part("venv", build_venv)

## Run all stages

The two GPUs run two independent chains at the same time.

- GPU 0: prep, base and instruct evaluation, SFT, DPO, reward model, best-of-n, distillation on teacher replies.
- GPU 1: GRPO, KTO, QLoRA, distillation on teacher logits.

A guard checks the SFT stage first: if more than half of its dev replies never stop, it prints raw replies and stops the stages that need the SFT model. A stage waits for the stages it needs. If a needed stage fails, the dependent stage is skipped and says so. Every training stage runs a probe of 4 steps first, measures the step time, and then sets its step count to fit its time budget (capped). DPO and KTO use 16 replies per step and the same number of steps. QLoRA and the teacher-reply run use the step count of the SFT run.

In [ ]:
# 5. The stage scripts (one file each) and the runner. Each stage is a subprocess with its own GPU, time limit, log and JSON result. A failed stage does not stop the others.
SCRIPTS = {
    "prep.py": r'''
# prep: download the weights, print the embedding evidence, fix the end token on a local copy of the base model, write the 50 GSM8K items
import json, torch
from common import *
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
for m in ("Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"): snapshot_download(m)
def evidence(label, path):  # is the <|im_end|> row of the (tied) embedding trained, or does it look like an unused row?
    t = AutoTokenizer.from_pretrained(path); E = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float32).get_input_embeddings().weight.detach()
    n = len(t); U = E[n:n + 50]; assert len(U) == 50, "no 50 unused rows"
    ids = {IM_END: t.convert_tokens_to_ids(IM_END), EOT: t.convert_tokens_to_ids(EOT), **{w: t.encode(w, add_special_tokens=False)[0] for w in (" the", " and", " water")}}
    cos = lambda i: float(torch.nn.functional.cosine_similarity(E[i][None], U).mean())
    rows = {k: (float(E[i].norm()), cos(i)) for k, i in ids.items()}; un = float(U.norm(dim=1).mean()); wn = sum(rows[w][0] for w in (" the", " and", " water")) / 3
    print(f"{label}: rows {n}..{n + 49} are unused. Mean norm of unused rows {un:.3f}, of 3 common words {wn:.3f}")
    for k, (nm, c) in rows.items(): print(f"   {k:16s} norm {nm:.3f}  mean cosine to the 50 unused rows {c:+.3f}")
    d = abs(rows[IM_END][0] - un) < abs(rows[IM_END][0] - wn)
    print(f"   -> the norm of {IM_END} is closer to the unused rows than to common words: {d}. Same test for {EOT}: {abs(rows[EOT][0] - un) < abs(rows[EOT][0] - wn)}", flush=True)
evidence("base", "Qwen/Qwen2.5-0.5B"); evidence("instruct", "Qwen/Qwen2.5-0.5B-Instruct")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B"); tok.eos_token, tok.pad_token = EOT, EOT   # a reply ends at <|endoftext|>: its row is trained in the base model
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", dtype=torch.float32)
model.generation_config.eos_token_id, model.generation_config.pad_token_id = eos_ids(tok), eos_ids(tok)[1]   # generation stops on either token
M.mkdir(parents=True, exist_ok=True); model.save_pretrained(M / "base05"); tok.save_pretrained(M / "base05")
from datasets import load_dataset
g = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=0).select(range(50))
json.dump([{"question": r["question"], "gold": float(r["answer"].split("####")[-1].strip().replace(",", ""))} for r in g], open(W / "gsm8k50.json", "w"))
print("prep ok: 3 models cached, base05 saved, gsm8k items:", len(g), flush=True)
''',
    "eval_two.py": r'''
# Part A: the same prompts on the base model and on the instruct model
import torch
from common import *
from transformers import AutoModelForCausalLM
tok = load_tok(M / "base05")
for name, path in (("base", str(M / "base05")), ("instruct", "Qwen/Qwen2.5-0.5B-Instruct")):
    m = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float32).cuda().eval()
    finish(name, m, tok, dict(params=sum(p.numel() for p in m.parameters())), sets=("dev", "held", "expr", "expr_direct", "gsm"))
    del m; torch.cuda.empty_cache()
''',
    "sft.py": r'''
# Parts B and F: supervised fine-tuning. Variants: lora (right replies from code), qlora (same, 4-bit base), teacher (replies from the 1.5B teacher)
import sys, json, torch
from common import *
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
variant = sys.argv[1]; tok = load_tok(M / "base05"); S = splits()
rows, extra = sft_rows(S["train"], 0), {}
if variant == "teacher":   # sequence-level distillation: the labels are the teacher's own replies, right or wrong
    from transformers import AutoModelForCausalLM
    t = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", dtype=torch.float32).cuda().eval()
    users = sorted({r["user"] for r in rows}); rep = dict(zip(users, (r for r, _, _ in gen(t, tok, [render(tok, u) for u in users], 80, 50))))
    rows = [dict(user=r["user"], completion=rep[r["user"]]) for r in rows]
    write("teacher", dict(stats=dict(params=sum(p.numel() for p in t.parameters())), replies=evaluate(t, tok)))
    del t; torch.cuda.empty_cache()
ds = Dataset.from_list([dict(prompt=render(tok, r["user"]), completion=r["completion"]) for r in rows])
def model_arg():
    if variant != "qlora": return str(M / "base05")
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    return AutoModelForCausalLM.from_pretrained(str(M / "base05"), dtype=torch.float32, device_map={"": 0}, quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float32))
def build(n):
    return SFTTrainer(model=model_arg(), train_dataset=ds, processing_class=tok, peft_config=lora(), args=cfg(
        SFTConfig, output_dir=str(M / "tmp"), max_steps=n, per_device_train_batch_size=16, learning_rate=2e-4, max_length=256, eos_token=EOT))
ref = W / "res" / "sft.json"   # the qlora and teacher runs use exactly the step count of the lora run
fixed = json.load(open(ref))["stats"]["steps"] if variant != "lora" and ref.exists() else None
tr, st = run_training(build, 480, 400, fixed)
if variant == "qlora":
    m = tr.accelerator.unwrap_model(tr.model).eval(); st["trainable"], st["total"] = m.get_nb_trainable_parameters()
    finish("qlora", m, tok, st, sets=("dev", "held"))
else:
    m, info = merged(tr, tok); st.update(info)
    finish("sft" if variant == "lora" else "distill_seq", m, tok, st, save=M / ("sft_merged" if variant == "lora" else "distill_seq_merged"))
''',
    "pref.py": r'''
# Part C: preference tuning from the SFT model. dpo = pairs (chosen, rejected). kto = single replies with a good or bad mark. Same replies per step, same steps.
import sys, json
from common import *
from datasets import Dataset
variant = sys.argv[1]; tok = load_tok(M / "base05"); P = pairs(splits()["train"], 1, 1000)
if variant == "dpo":
    from trl import DPOConfig as C, DPOTrainer as T
    ds = Dataset.from_list([dict(prompt=render(tok, p["prompt"]), chosen=p["chosen"], rejected=p["rejected"]) for p in P]); kw = dict(per_device_train_batch_size=4)
else:
    from trl import KTOConfig as C, KTOTrainer as T
    ds = Dataset.from_list([dict(prompt=render(tok, r["user"]), completion=r["completion"], label=r["label"]) for r in kto_rows(P, 0)]); kw = dict(per_device_train_batch_size=8)
def build(n):
    return T(model=str(M / "sft_merged"), train_dataset=ds, processing_class=tok, peft_config=lora(), args=cfg(
        C, output_dir=str(M / "tmp"), max_steps=n, gradient_accumulation_steps=2, learning_rate=2e-5, beta=0.1, max_length=320, **kw))
tr, st = run_training(build, 540, 100)    # 100 steps x 16 replies = 1,600 replies in both methods
m, info = merged(tr, tok); st.update(info)
finish(variant, m, tok, st)
''',
    "rm.py": r'''
# Part D: reward model 1 on the 1,000 confounded pairs (the DPO pairs). Reward model 2 on 1,000 balanced pairs (K1, K2 and K3 in equal parts)
from common import *
from datasets import Dataset
from trl import RewardConfig, RewardTrainer
tok = load_tok(M / "base05"); items = splits()["train"]
def train(name, P, budget):
    ds = Dataset.from_list([dict(prompt=render(tok, p["prompt"]), chosen=p["chosen"], rejected=p["rejected"]) for p in P])
    def build(n):
        return RewardTrainer(model=str(M / "base05"), train_dataset=ds, processing_class=tok, peft_config=lora("SEQ_CLS"), args=cfg(
            RewardConfig, output_dir=str(M / "tmp"), max_steps=n, per_device_train_batch_size=8, gradient_accumulation_steps=2, learning_rate=1e-4, max_length=320))
    tr, st = run_training(build, budget, 150); st["kinds"] = {k: sum(p["kind"] == k for p in P) for k in KINDS}
    tr.save_model(str(M / f"{name}_adapter")); write(name, dict(stats=st))
train("rm", pairs(items, 1, 1000), 240)
try: train("rm_bal", pairs(items, 2, 1000, ("K1", "K2", "K3")), 200)
except Exception as e: print("balanced reward model failed:", repr(e)[:300], flush=True)
''',
    "bon.py": r'''
# Part D: best-of-n against the reward models. 16 samples per prompt from the SFT policy, scored by each reward model and by the exact grader
import torch
from common import *
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification
tok = load_tok(M / "base05"); S = splits(); items = S["dev"] + S["held"]; N = 16
pol = AutoModelForCausalLM.from_pretrained(str(M / "sft_merged"), dtype=torch.float32).cuda().eval()
raw = gen(pol, tok, [render(tok, s["user"]) for s in items], 80, bs=16, n=N, temp=1.0)
samples = [[dict(reply=r, cut=c) for r, c, _ in raw[i * N:(i + 1) * N]] for i in range(len(items))]
del pol; torch.cuda.empty_cache(); tok.padding_side = "right"
HP = held_pairs(S["held"]); us = [p["prompt"] for p in HP]
def score_with(adapter):
    rm = AutoModelForSequenceClassification.from_pretrained(str(M / "base05"), num_labels=1, dtype=torch.float32)
    rm.config.pad_token_id = eos_ids(tok)[1]; rm = PeftModel.from_pretrained(rm, str(M / adapter)).cuda().eval()
    def score(users, replies, bs=32):   # same text as the training pairs: prompt, reply, end token
        out = []
        for i in range(0, len(users), bs):
            enc = tok([render(tok, u) + r + EOT for u, r in zip(users[i:i + bs], replies[i:i + bs])], return_tensors="pt", padding=True).to("cuda")
            with torch.no_grad(): out += rm(**enc).logits[:, 0].float().tolist()
        return out
    d = dict(rm=[score([s["user"]] * N, [x["reply"] for x in sm]) for s, sm in zip(items, samples)], pair_chosen=score(us, [p["chosen"] for p in HP]),
             pair_rejected=score(us, [p["rejected"] for p in HP]), probes={k: score([s["user"] for s in S["held"]], [v] * 120) for k, v in PROBES.items()})
    del rm; torch.cuda.empty_cache(); return d
out = dict(stats={}, samples=samples, n=N, pair_kinds=[p["kind"] for p in HP], **score_with("rm_adapter"))
if (M / "rm_bal_adapter").exists():
    try: out["bal"] = score_with("rm_bal_adapter")
    except Exception as e: print("balanced reward model scoring failed:", repr(e)[:300], flush=True)
write("bon", out)
''',
    "grpo.py": r'''
# Part E: GRPO on arithmetic from the base model. The reward is the code in expr_reward. No reward model, no critic.
from common import *
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
tok = load_tok(M / "base05"); E = expr_split()
ds = Dataset.from_list([dict(prompt=render(tok, s["user"]), gold=s["g"]) for s in E["train"][:2000]])
def reward(prompts, completions, gold, **kw): return [expr_reward(c, g) for c, g in zip(completions, gold)]
def build(n):
    return GRPOTrainer(model=str(M / "base05"), reward_funcs=reward, train_dataset=ds, processing_class=tok, peft_config=lora(r=32), args=cfg(
        GRPOConfig, output_dir=str(M / "tmp"), max_steps=n, per_device_train_batch_size=16, gradient_accumulation_steps=2, num_generations=8,
        max_completion_length=96, temperature=1.0, learning_rate=3e-5, beta=0.0))
tr, st = run_training(build, 900, 200)
m, info = merged(tr, tok); st.update(info)
finish("grpo", m, tok, st)
''',
    "distill.py": r'''
# Part F: logit distillation with TRL's DistillationTrainer. The student writes replies, the 1.5B teacher scores every token of them.
import torch
from common import *
from datasets import Dataset
from transformers import AutoModelForCausalLM
from trl import DistillationConfig, DistillationTrainer
tok = load_tok(M / "base05"); S = splits()
start = M / "distill_seq_merged"; start = start if start.exists() else M / "base05"   # continue from the student of the sequence-level run
ds = Dataset.from_list([dict(prompt=render(tok, s["user"])) for s in S["train"][:600]])
# The teacher ends replies with <|im_end|>. The student ends them with <|endoftext|> (the trained row). Swap the two output rows of the teacher, so its end
# probability sits on <|endoftext|>. The teacher gets its own untied output matrix first, so its input embeddings stay as they were.
teacher = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", dtype=torch.float32)
w = teacher.get_output_embeddings().weight.detach().clone(); a, b = eos_ids(tok); w[[a, b]] = w[[b, a]]
teacher.config.tie_word_embeddings = False; teacher.lm_head = torch.nn.Linear(w.shape[1], w.shape[0], bias=False); teacher.lm_head.weight = torch.nn.Parameter(w)
def build(n):
    return DistillationTrainer(model=str(start), teacher_model=teacher, train_dataset=ds, processing_class=tok, peft_config=lora(), args=cfg(
        DistillationConfig, output_dir=str(M / "tmp"), max_steps=n, per_device_train_batch_size=8, max_completion_length=80, learning_rate=5e-5))
tr, st = run_training(build, 420, 100); st["start"] = start.name
m, info = merged(tr, tok); st.update(info)
finish("distill_logit", m, tok, st)
''',
}
for n, src in SCRIPTS.items(): ast.parse(src); (W / "stages" / n).write_text(src)
STAGE = {}                                                    # name -> (status, seconds, note)
def tail(name, n=12):
    p = W / "logs" / f"{name}.log"
    return "\n".join(l[:190] for l in (p.read_text(errors="replace").splitlines()[-n:] if p.exists() else []))

def guard_sft():  # a fault like a bad end token shows here, minutes after the SFT stage, not at the end of the notebook
    r = json.load(open(W / "res" / "sft.json")); rows = r["replies"]["dev"]; o = [outcome(s, x) for s, x in zip(splits()["dev"], rows)]
    if o.count("cut") * 2 > len(o):
        print(f"GUARD: {o.count('cut')} of {len(o)} SFT dev replies never stop. Raw replies:"); [print("  ", repr(x["reply"][:100])) for x in rows[:3]]
        return f"guard: {o.count('cut')} of {len(o)} dev replies are cut, so the stages that need the SFT model are stopped"
GUARDS = {"sft": guard_sft}
def run_stage(name, script, args=(), gpu=0, limit=900, needs=("prep",), outs=None, wait=1500):
    t0 = time.time()
    while any(STAGE.get(n, ("pending",))[0] in ("pending", "running") for n in needs) and time.time() - t0 < wait: time.sleep(3)
    bad = [n for n in needs if STAGE.get(n, ("pending",))[0] != "ok"]
    left = DEADLINE - time.time() - 150
    if bad or left < 90:
        STAGE[name] = ("skipped", 0, f"needs {bad}" if bad else "no time left"); lap(f"{name:13s} skipped: {STAGE[name][2]}"); return
    lim = min(limit, left); STAGE[name] = ("running", 0, ""); t1 = time.time()
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH=str(W), ALIGN_DIR=str(W), ALIGN_MODELS=str(M), PYTHONUNBUFFERED="1",
               TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_TELEMETRY="1", HF_HUB_DOWNLOAD_TIMEOUT="60")
    with open(W / "logs" / f"{name}.log", "w") as f:
        p = subprocess.Popen([str(PY), str(W / "stages" / script), *args], cwd=W, env=env, stdout=f, stderr=subprocess.STDOUT, start_new_session=True)
        try: rc = p.wait(timeout=lim); st = "ok" if rc == 0 else f"failed rc={rc}"
        except subprocess.TimeoutExpired: os.killpg(p.pid, signal.SIGKILL); p.wait(); st = f"timeout after {lim:.0f} s"
    missing = [o for o in (outs or [name]) if not (W / "res" / f"{o}.json").exists()] if name != "prep" else []
    if st == "ok" and missing: st = f"failed: no result file {missing}"
    if st == "ok" and name in GUARDS: st = GUARDS[name]() or st
    STAGE[name] = (st, round(time.time() - t1), "" if st == "ok" else tail(name))
    lap(f"{name:13s} gpu{gpu} {st} in {STAGE[name][1]} s")
    if st != "ok": print(STAGE[name][2])

# name, script, args, gpu, limit (s), needs, result files
PLAN = [("prep", "prep.py", (), 0, 900, (), None),
        ("base", "eval_two.py", (), 0, 900, ("prep",), ["base", "instruct"]), ("sft", "sft.py", ("lora",), 0, 1200, ("prep",), None),
        ("dpo", "pref.py", ("dpo",), 0, 1200, ("prep", "sft"), None), ("rm", "rm.py", (), 0, 700, ("prep",), None),
        ("bon", "bon.py", (), 0, 900, ("prep", "sft", "rm"), None), ("distill_seq", "sft.py", ("teacher",), 0, 1500, ("prep", "sft"), ["distill_seq", "teacher"]),
        ("grpo", "grpo.py", (), 1, 1800, ("prep",), None), ("kto", "pref.py", ("kto",), 1, 1200, ("prep", "sft"), None),
        ("qlora", "sft.py", ("qlora",), 1, 700, ("prep", "sft"), None), ("distill_logit", "distill.py", (), 1, 1200, ("prep", "distill_seq"), None)]
def run_all():
    (W / "res").mkdir(exist_ok=True)
    def chain(gpu):
        for n, sc, a, g, li, ne, ou in PLAN:
            if g == gpu and n != "prep": run_stage(n, sc, a, g, li, ne, ou)
    run_stage(*PLAN[0][:6])
    ts = [threading.Thread(target=chain, args=(g,)) for g in (0, 1)]
    for t in ts: t.start()
    for t in ts: t.join()

In [ ]:
# 6. Run every stage. Chain 0 uses GPU 0 and chain 1 uses GPU 1, so two independent stages run at the same time. This cell takes about an hour.
part("run all stages", run_all)
print(pd.DataFrame([dict(stage=n, status=v[0], seconds=v[1]) for n, v in STAGE.items()]).to_string(index=False))

In [ ]:
# 7. Load the stage results and define the scoring helpers. Every number below is graded here, from the raw replies, by the code from cell 2.
ALL = ["base", "instruct", "sft", "qlora", "dpo", "kto", "rm", "bon", "grpo", "teacher", "distill_seq", "distill_logit"]
S, E = splits(), expr_split()
ITEMS = {"dev": S["dev"], "held": S["held"], "expr": E["eval"], "expr_direct": E["eval"]}
GSM = json.load(open(W / "gsm8k50.json")) if (W / "gsm8k50.json").exists() else []
R = {}
def load_results():
    R.clear(); R.update({p.stem: json.load(open(p)) for p in sorted((W / "res").glob("*.json"))})
    bad = [n for n, r in R.items() if r["data_hash"] != data_hash()]
    assert not bad, f"stage and kernel disagree on the data: {bad}"
    print("results found:", sorted(R)); print("not reached or failed:", [n for n in ALL if n not in R])
def stage_facts():
    rows = [dict(stage=n, steps=r["stats"].get("steps"), probe_s_per_step=r["stats"].get("probe_t_step"), s_per_step=r["stats"].get("t_step"), train_s=r["stats"].get("train_s"),
                 budget_s=r["stats"].get("budget_s"), peak_GiB=r["stats"].get("peak_gib"), loss_finite=r["stats"].get("nan_free")) for n, r in R.items() if "steps" in r["stats"]]
    show("stage_facts", pd.DataFrame(rows))
def have(*ns): return all(n in R for n in ns)
def outs(n, ds="held"): return [outcome(s, r) for s, r in zip(ITEMS[ds], R[n]["replies"][ds])]
def oks(n, ds="held"): return [o == "pass" for o in outs(n, ds)]
def chars(n, ds="held"): rows = R[n]["replies"][ds]; return sum(len(r["reply"]) for r in rows) / len(rows)
def gsm_oks(n): return [gsm_ok(r, g["gold"]) for r, g in zip(R[n]["replies"].get("gsm", []), GSM)]   # [] when the stage did not run GSM8K
def gsm_txt(n, base="base"):
    k = len(gsm_oks(n)); return "n/a" if not k else (f"{sum(gsm_oks(n))}/{k}" + (f" ({sum(gsm_oks(n)) - sum(gsm_oks(base)):+d})" if base != n and gsm_oks(base) else ""))
def raw_replies(names, ds="held", k=2):  # raw text, 80 characters, with the stop token that ended it
    for n in names:
        for r in (R[n]["replies"][ds][:k] if have(n) else []):
            print(f"   {n:13s} [{'CUT: no stop token' if r['cut'] else 'stop ' + (r.get('stop') or '?')}] {r['reply'][:80]!r}")
def line(n, ds="held"):
    o = outs(n, ds)
    return dict(model=n, n=len(o), **{"pass": rate(o.count("pass"), len(o))}, wrong=o.count("wrong"), cut=o.count("cut"), empty=o.count("empty"), mean_chars=round(chars(n, ds)))
CONSTS = ["Answer: 0", "OK", '{"name": "Anna"}', "a\nb\nc", "the moon is old.", "Note: the moon is old."]
def const_line(ds="held"):  # baseline: the best single reply that is the same for every prompt
    items = ITEMS[ds]; cands = CONSTS if ds in ("dev", "held") else [f"Answer: {g}" for g in {s["g"] for s in items}]
    k, c = max((sum(check(s, c) for s in items), c) for c in cands)
    return dict(model=f"always {c[:12]!r}", n=len(items), **{"pass": rate(k, len(items))}, wrong=len(items) - k, cut=0, empty=0, mean_chars=len(c))
def pair_test(a, b, ds="held", label=""):  # exact McNemar test of a against b on the same items
    x, y, p = mcnemar(oks(a, ds), oks(b, ds)); v = verdict(x - y, p)
    print(f"{label or ds}: {a} only right {x}, {b} only right {y}, exact p={p:.4f} -> " + (f"no reliable difference between {a} and {b}" if v.startswith("no") else f"{a} is {v} than {b}"))
def curves(names, key, fname, title):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for n in names:
        pts = [(e["step"], e[key]) for e in R[n]["stats"]["log"] if key in e and "step" in e] if have(n) else []
        if pts: ax.plot(*zip(*pts), label=n)
    ax.set_xlabel("step"); ax.set_ylabel(key); ax.set_title(title); ax.grid(alpha=.3); ax.legend(); fig.tight_layout(); fig.savefig(OUT / fname, dpi=110); plt.close(fig)
part("load results", load_results); part("stage facts", stage_facts)

## Part A. Base against instruct

Question: what is the difference between a base model and an instruction-tuned model, when both see the same prompt?

Why the exam asks: `models.7` lists base and instruction-tuned models. The base model has seen no instruction tuning, so it often does not stop. The deterministic grader marks that as failure. A second table gives the same arithmetic with and without room to think. This shows what thinking tokens cost and buy in a model that is not a reasoning model.

In [ ]:
# A1. Base against instruct: the same 40 dev instructions, then the 120 held-out ones. A reply that never stops is "cut". It is never a pass.
def part_a():
    show("A_dev40", pd.DataFrame([line(m, "dev") for m in ("base", "instruct")] + [const_line("dev")]))
    show("A_held120", pd.DataFrame([line(m) for m in ("base", "instruct")] + [const_line()]))
    pair_test("instruct", "base", "dev", "dev, 40 items"); pair_test("instruct", "base", "held", "held, 120 items")
    fam = pd.DataFrame({m: {f: sum(o for s, o in zip(S["held"], oks(m)) if s["fam"] == f) for f in FAMS} for m in ("base", "instruct")})
    show("A_per_family_pass_of_20", fam.reset_index(names="family"))
    print("first reply of each model to the first held prompt:"); [print(f"  {m:9s}", repr(R[m]["replies"]["held"][0]["reply"][:90]), "cut" if R[m]["replies"]["held"][0]["cut"] else "") for m in ("base", "instruct")]
# A2. Reasoning tokens: the same 100 arithmetic items with no room to think, and with room to think. The final line is graded alike.
def part_a2():
    rows = []
    for m in ("base", "instruct"):
        for ds, how in (("expr_direct", "answer only"), ("expr", "think first")):
            rows.append(dict(line(m, ds), prompt=how, well_formed=sum(last_answer(r["reply"]) is not None for r in R[m]["replies"][ds])))
    show("A2_think_first", pd.DataFrame(rows + [dict(const_line("expr"), prompt="same reply always", well_formed=100)]))
    for m in ("base", "instruct"):
        o = outs(m, "expr_direct"); true_ok = sum(check(s, "Answer: %d" % s["g"]) for s in E["eval"])
        print(f"{m}, answer only: right {o.count('pass')}, well-formed but wrong {o.count('wrong')}, cut {o.count('cut')}. Grader on the true answers of the same items: {true_ok}/100 pass")
    print("raw replies, answer only (true answers", [s["g"] for s in E["eval"][:2]], "):"); raw_replies(["base", "instruct"], "expr_direct")
    print("raw replies, think first:"); raw_replies(["base", "instruct"], "expr")
part("A base vs instruct", part_a); part("A2 thinking tokens", part_a2)

## Part B. Instruction tuning with SFT, LoRA and QLoRA

Question: how much of the instruct model's behaviour does a small SFT run on 1,500 code-made examples give the base model?

Why the exam asks: `customization.3` names SFT, PEFT, LoRA, QLoRA and instruction tuning. The tables show the share of trainable parameters, the size of the adapter, the proof that merging the adapter does not change the model, and the memory of the 4-bit run.

In [ ]:
# B1. Instruction tuning (SFT with LoRA): before and after on the 120 held-out instructions. The QLoRA run uses the same data, steps and seed.
def part_b():
    show("B_instruction_following", pd.DataFrame([line(n) for n in ("base", "sft", "qlora", "instruct") if have(n)] + [const_line()]))
    pair_test("sft", "base")
    if have("qlora"): pair_test("qlora", "sft")
    ns = [n for n in ("base", "sft", "instruct") if have(n)]
    show("B_per_family_pass_of_20", pd.DataFrame({m: {f: sum(o for s, o in zip(S["held"], oks(m)) if s["fam"] == f) for f in FAMS} for m in ns}).reset_index(names="family"))
    rows = []
    for n in ("sft", "qlora"):
        if not have(n): continue
        st = R[n]["stats"]
        rows.append(dict(run=n, steps=st["steps"], s_per_step=st["t_step"], train_s=st["train_s"], peak_GiB=st["peak_gib"], trainable=st["trainable"], pct_trainable=round(100 * st["trainable"] / st["total"], 2),
                         adapter_MB=round(st["trainable"] * 4 / 2**20, 1), merge_logit_diff=f"{st.get('merge_max_logit_diff', float('nan')):.1e}", loss_finite=st["nan_free"]))
    show("B_peft_facts", pd.DataFrame(rows))
    print("raw replies on the first held prompt:"); raw_replies(["sft", "qlora"])
    curves(["sft", "qlora"], "loss", "B_sft_loss.png", "SFT training loss")
part("B instruction tuning", part_b)

## Part C. Preference data: DPO and KTO

Question: what does each method need, and what does it change when both start from the SFT model and see the same number of replies?

Why the exam asks: DPO needs pairs. KTO needs single replies with a good or bad mark. The same labels feed both. The win rate uses the exact rule, so it needs no judge.

In [ ]:
# C1. DPO and KTO start from the SFT model. Win rate: per held prompt, the new reply against the SFT reply under the exact rule (right beats wrong, then shorter beats longer).
def key(s, r): ok = outcome(s, r) == "pass"; return (ok, -len(r["reply"]) * ok)
def wtl(n, ref):
    w = l = t = 0
    for s, a, b in zip(S["held"], R[n]["replies"]["held"], R[ref]["replies"]["held"]):
        ka, kb = key(s, a), key(s, b); w += ka > kb; l += ka < kb; t += ka == kb
    from scipy.stats import binomtest
    return w, t, l, (binomtest(w, w + l, 0.5).pvalue if w + l else 1.0)
def part_c():
    ns = [n for n in ("sft", "dpo", "kto") if have(n)]; rows = []
    for n in ns:
        o = outs(n); w, t, l, p = wtl(n, "sft")
        rows.append(dict(model=n, steps=R[n]["stats"]["steps"], right=rate(o.count("pass"), 120), mean_chars=round(chars(n)), cut=o.count("cut"), win_vs_sft=rate(int(round(w + t / 2)), 120),
                         W_T_L=f"{w}/{t}/{l}", sign_p=round(p, 4), train_s=R[n]["stats"]["train_s"], peak_GiB=R[n]["stats"]["peak_gib"]))
    show("C_dpo_kto", pd.DataFrame(rows)); print("win rate: a tie counts half. Random baseline 0.50. Baseline for right: the best same-for-every-prompt reply gives", const_line()["pass"])
    for n in ("dpo", "kto"):
        if have(n): pair_test(n, "sft")
    if have("dpo", "kto"):
        pair_test("dpo", "kto"); print("same budget:", R["dpo"]["stats"]["steps"] == R["kto"]["stats"]["steps"], "| replies per step: 16 in both")
    for n in ("dpo", "kto"):
        if have(n): print(n, "last logged reward metrics:", {k: round(v, 3) for k, v in {k: v for e in R[n]["stats"]["log"] for k, v in e.items() if "reward" in k}.items()})
    print("raw replies on the first held prompt:"); raw_replies(["dpo", "kto"])
    curves(["dpo", "kto"], "loss", "C_dpo_kto_loss.png", "Preference tuning loss")
part("C DPO and KTO", part_c)

## Part D. RLHF and reward hacking

Question: a reward model learns from pairs. What does it learn when length always points to the chosen reply, and what happens when we optimise against it?

Reward model 1 trains on the confounded pairs. Reward model 2 trains on 1,000 balanced pairs (K1, K2 and K3 in equal parts), with the same recipe. Why the exam asks: RLHF needs a reward model, and a reward model is only a proxy. The data decides what it learns. Best-of-n is the simplest way to optimise against it. The code grader gives the true score. The size n is chosen on the dev items and reported on the held-out items.

In [ ]:
# D1. RLHF: reward model 1 (trained on K1 and K2 pairs only, chosen always shorter) and reward model 2 (balanced pairs, K3 included), then best-of-n. Truth = the code grader.
def rm_tables(tag, b, N):
    from scipy.stats import spearmanr
    kinds = np.array(b["pair_kinds"]); ch, rj = np.array(b["pair_chosen"]), np.array(b["pair_rejected"])
    what = {"K1": "right-short over right-long", "K2": "right-short over wrong-long", "K3": "right-long over wrong-short"}
    show(f"D_{tag}_pair_accuracy", pd.DataFrame([dict(kind=k, what=what[k], n=int((kinds == k).sum()), accuracy=rate(int((ch[kinds == k] > rj[kinds == k]).sum()), int((kinds == k).sum())), random="0.50") for k in what]))
    items = S["dev"] + S["held"]; rm = np.array(b["rm"]); ok = np.array([[outcome(s, x) == "pass" for x in sm] for s, sm in zip(items, b["samples"])])
    ln = np.array([[len(x["reply"]) for x in sm] for sm in b["samples"]])
    print(f"{tag}: Spearman(reward, length) {spearmanr(rm.ravel(), ln.ravel())[0]:.2f} | Spearman(reward, right) {spearmanr(rm.ravel(), ok.ravel())[0]:.2f} | samples right {ok.mean():.2f}"
          f" | among right replies, Spearman(reward, length) {spearmanr(rm.ravel()[ok.ravel()], ln.ravel()[ok.ravel()])[0]:.2f}")
    rows = []
    for split, (lo, hi) in {"dev": (0, 40), "held": (40, 160)}.items():
        idx = np.arange(hi - lo)
        for n in (1, 2, 4, 8, 16):
            sel = rm[lo:hi, :n].argmax(1)
            rows.append(dict(split=split, n=n, mean_reward=round(float(rm[lo:hi][idx, sel].mean()), 2), right=rate(int(ok[lo:hi][idx, sel].sum()), hi - lo), mean_chars=round(float(ln[lo:hi][idx, sel].mean())),
                             oracle_pass_at_n=rate(int(ok[lo:hi, :n].any(1).sum()), hi - lo)))
    df = pd.DataFrame(rows); show(f"D_{tag}_best_of_n", df)
    dv = df[df.split == "dev"]; best = int(dv.loc[dv["right"].map(lambda x: float(x.split()[0])).idxmax(), "n"])   # n chosen on dev only
    hv = df[(df.split == "held") & (df.n == best)].iloc[0]; h1 = df[(df.split == "held") & (df.n == 1)].iloc[0]
    print(f"{tag}: n chosen on dev = {best}. Held, n=1 (random baseline): right {h1['right']}, mean reward {h1['mean_reward']}, chars {h1['mean_chars']}. Held, n={best}: right {hv['right']}, mean reward {hv['mean_reward']}, chars {hv['mean_chars']}.")
    gold = ch[kinds == "K1"]
    show(f"D_{tag}_junk_replies", pd.DataFrame([dict(junk_reply=repr(common.PROBES[k]), true_right=sum(check(s, common.PROBES[k]) for s in S["held"]), mean_reward=round(float(np.mean(v)), 2),
                                                    beats_gold_short=rate(int((np.array(v) > gold).sum()), 120)) for k, v in b["probes"].items()]
                                                   + [dict(junk_reply="gold short reply", true_right=120, mean_reward=round(float(gold.mean()), 2), beats_gold_short="-")]))
    k3 = (ch[kinds == "K3"] > rj[kinds == "K3"]).mean(); print(f"{tag}: K3 accuracy {k3:.2f}. The true rule gives 1.00. It prefers the wrong-short reply in {1 - k3:.2f} of K3 pairs.")
def part_d():
    b = R["bon"]; N = b["n"]; rm_tables("rm_confounded", b, N)
    st = R["rm"]["stats"]; print("confounded reward model: training pair kinds", st.get("kinds"), "| steps", st["steps"], "| s per step", st["t_step"])
    if "bal" in b: rm_tables("rm_balanced", dict(b, **b["bal"]), N); print("balanced reward model: training pair kinds", R["rm_bal"]["stats"].get("kinds") if have("rm_bal") else "n/a", "| same size (1,000 pairs), same recipe")
    else: print("balanced reward model: not available")
    curves(["rm", "rm_bal"], "loss", "D_reward_model_loss.png", "Reward model loss")
part("D RLHF and reward hacking", part_d)

## Part E. GRPO

Question: can a model learn arithmetic from a reward that code checks, with no reward model and no labelled replies?

Why the exam asks: GRPO fits tasks where the answer can be checked. The reward is 1 for the right last line, 0.1 for a well-formed wrong one, and 0 otherwise. Only the last line counts, so a reply that lists many numbers earns nothing.

In [ ]:
# E1. GRPO: 8 replies per prompt, reward = code. Arithmetic accuracy on 100 held-out items before (the base model) and after.
def part_e():
    st = R["grpo"]["stats"]; rw = [(e["step"], e["reward"]) for e in st["log"] if "reward" in e]
    k = max(1, len(rw) // 5); print(f"{st['steps']} steps, {st['t_step']} s per step. Mean reward of the first {k} logs: {np.mean([r for _, r in rw[:k]]):.3f}. Of the last {k}: {np.mean([r for _, r in rw[-k:]]):.3f}. Max possible: 1.0")
    curves(["grpo"], "reward", "E_grpo_reward.png", "GRPO mean reward per step")
    rows = []
    for n in ("base", "grpo"):
        rows.append(dict(line(n, "expr"), no_answer_line=sum(last_answer(r["reply"]) is None for r in R[n]["replies"]["expr"])))
    show("E_grpo_arithmetic", pd.DataFrame(rows + [dict(const_line("expr"), no_answer_line=0)])); pair_test("grpo", "base", "expr", "arithmetic, 100 items")
    print("raw replies, arithmetic:"); raw_replies(["base", "grpo"], "expr")
    show("E_grpo_side_effects", pd.DataFrame([line(n) for n in ("base", "grpo")])); pair_test("grpo", "base", "held", "instructions, 120 items")
part("E GRPO", part_e)

## Part F. Distillation

Question: how much does a 0.5B student get from a 1.5B teacher, and does matching the teacher's token probabilities add to copying its replies?

Why the exam asks: distillation trains a small model from a larger one. The teacher ends replies with `<|im_end|>` and the student with `<|endoftext|>`. Before the logit run, the two output rows of the teacher are swapped (on an untied copy of its output matrix), so its end probability lands on `<|endoftext|>`. The student is compared with the teacher and with SFT on code-made replies.

In [ ]:
# F1. Distillation. SFT = right replies from code. distill_seq = SFT on the teacher's replies. distill_logit = then the teacher scores every token of the student's own replies.
def part_f():
    ns = [n for n in ("base", "sft", "distill_seq", "distill_logit", "teacher", "instruct") if have(n)]; rows = []
    for n in ns:
        r = line(n); r["arith_right"] = rate(sum(oks(n, "expr")), 100); r["gsm8k_right"] = rate(sum(gsm_oks(n)), 50) if gsm_oks(n) else "n/a"; rows.append(r)
    show("F_distillation", pd.DataFrame(rows + [const_line()]))
    if have("teacher", "base"): print(f"teacher parameters {R['teacher']['stats']['params'] / 1e6:.0f} M, student parameters {R['base']['stats']['params'] / 1e6:.0f} M")
    for a, b in (("distill_seq", "sft"), ("distill_logit", "distill_seq"), ("teacher", "distill_logit"), ("teacher", "sft")):
        if have(a, b): pair_test(a, b)
    if have("distill_logit"): print("logit run started from:", R["distill_logit"]["stats"]["start"], "| steps", R["distill_logit"]["stats"]["steps"], "| s per step", R["distill_logit"]["stats"]["t_step"], "| peak GiB", R["distill_logit"]["stats"]["peak_gib"])
    print("raw replies on the first held prompt:"); raw_replies(["distill_seq", "distill_logit", "teacher"])
    curves(["distill_seq", "distill_logit"], "loss", "F_distill_loss.png", "Distillation training loss")
part("F distillation", part_f)

## Part G. Summary

Question: one table for all methods: what data, what loss, what time, what memory, what gain, what side effect, what forgetting?

In [ ]:
# G1. One table. Gain = change of the right-rate on the held-out items against the starting model (stated in the column "from"). Forgetting = change on 50 GSM8K items against the base model.
def part_g():
    spec_ = [("SFT (LoRA)", "sft", "base", "held", "prompt + right reply", "next-token loss on the reply"), ("SFT (QLoRA)", "qlora", "base", "held", "prompt + right reply", "same, base in 4 bit"),
             ("DPO", "dpo", "sft", "held", "prompt + chosen + rejected", "sigmoid of reward margin"), ("KTO", "kto", "sft", "held", "prompt + reply + good/bad mark", "prospect-theory utility"),
             ("GRPO", "grpo", "base", "expr", "prompt + code reward", "group-normalised reward"), ("distill: teacher replies", "distill_seq", "sft", "held", "prompt + teacher reply", "next-token loss on teacher reply"),
             ("distill: teacher logits", "distill_logit", "distill_seq", "held", "prompt + teacher scores", "reverse KL to the teacher")]
    rows = []
    for label, n, ref, ds, data, loss in spec_:
        if not have(n, ref): rows.append(dict(method=label, note="not reached or failed")); continue
        st = R[n]["stats"]; x, y, p = mcnemar(oks(n, ds), oks(ref, ds));         rows.append(dict(method=label, data=data, loss_or_reward=loss, train_s=st["train_s"], peak_GiB=st["peak_gib"], on=ds, gain=f"{(x - y) / len(oks(n, ds)):+.2f} vs {ref}", p=round(p, 3),
                         chars=f"{round(chars(n, ds) - chars(ref, ds)):+d}", gsm_right=gsm_txt(n)))
    show("G_summary", pd.DataFrame(rows).fillna("")); print("baseline for held rows: the best same-for-every-prompt reply gives", const_line()["pass"], "| arithmetic:", const_line("expr")["pass"])
    print(f"GSM8K reference: base {gsm_txt('base')}, instruct {gsm_txt('instruct') if have('instruct') else 'n/a'}. Paired p per row: exact McNemar on the same items. chars = change in mean reply length.")
    if have("bon"):
        print("RLHF row: no policy weights change. Best-of-n numbers are in table D_best_of_n. Reward model training:", R["rm"]["stats"]["train_s"] if have("rm") else "n/a", "s")
    fig, ax = plt.subplots(figsize=(7, 3.5)); ns = [n for n in ("base", "sft", "qlora", "dpo", "kto", "grpo", "distill_seq", "distill_logit", "instruct") if have(n)]
    ax.bar(ns, [sum(gsm_oks(n)) / 50 if gsm_oks(n) else float("nan") for n in ns]); ax.set_ylabel("GSM8K right of 50"); ax.tick_params(axis="x", rotation=45); fig.tight_layout(); fig.savefig(OUT / "G_gsm8k_forgetting.png", dpi=110); plt.close(fig)
part("G summary", part_g)

## What to look for

- **Rates.** Every rate has a Wilson 95% interval. With 120 items, one item is 0.8 points. With 40 items, one item is 2.5 points. Read gaps inside the interval as noise.
- **Paired tests.** Two systems on the same items use an exact McNemar test. The printed verdict comes from the p value and the sign of the gain. It is not a written conclusion.
- **Cut, empty and wrong.** These are separate outcomes. A model that never stops is cut. Read `cut` next to `pass`.
- **Baselines.** The row "always ..." is the best single reply that is the same for every prompt. A win rate of 0.50 is the random baseline.
- **Length.** `mean_chars` shows the side effect of the preference methods. Training pairs always have the shorter reply as the chosen one.
- **Reward model.** Compare accuracy on K1 and K2 with K3. The gap shows how much it learned from length. Then read the best-of-n table: the reward rises with n by construction. The true score is the right-rate.
- **GRPO.** Read the first and last mean reward and the held-out accuracy. A reward gain with no accuracy gain would be hacking.
- **Forgetting.** 50 GSM8K items: one item is 2 points. A small gap is noise.
- **Limits.** One seed per run. One 0.5B model. LoRA only. Held-out sets are small. The reward model has 1,000 pairs. The distillation student is short-trained. PPO is not run: best-of-n stands in for the RL step of RLHF. The teacher runs in fp32.

## Exam rules this proves

1. `customization.3` SFT: SFT trains on prompt and right-reply pairs. Part B measures what it does to a base model.
2. `customization.3` PEFT, LoRA, QLoRA: they train a small share of the weights. Part B prints the share, the adapter size, the memory of the 4-bit base, and the check that merging keeps the logits.
3. `customization.3` DPO, KTO: DPO needs pairs. KTO needs single replies with a good or bad mark. Part C trains both from the same labels with the same budget.
4. `customization.3` RLHF: a reward model is a proxy. Part D shows the gap between its reward and the true score.
5. `customization.3` GRPO: with a checkable answer, a reward function replaces the reward model. Part E trains with it.
6. `customization.3` distillation: a student learns from a teacher by replies or by token probabilities. Part F does both.
7. `models.7` base against instruction-tuned: Part A. Foundation against task-specific: Part G (gain on the task against loss on GSM8K).

| Exam term | One plain line | Source file | Measured in |
|---|---|---|---|
| Fine-tuning methods | Ways to change a pretrained model with new data. | `ai-training.txt` (post-training) | G |
| SFT | Train on prompt and right-reply pairs. | `mastering-agentic-techniques-ai-agent-customization.txt` | B |
| PEFT | Freeze most weights. Train a tiny share. | same | B |
| LoRA | Add small trainable matrices to attention and feed-forward layers. | same | B |
| QLoRA | LoRA on a base model with quantized linear layers. | `qlora.txt` | B |
| RLHF | Learn from feedback with reinforcement learning. A reward model comes first. | `selecting-large-language-model-customization-techniques.txt` | D (best-of-n, no PPO) |
| Preference optimization | Train on which of two replies is better. | agentic source | C |
| DPO | Train on chosen and rejected pairs. No separate reward model. | agentic source | C |
| KTO | Train on single replies with a good or bad mark. | `gtc24-s62092.txt` | C |
| GRPO | Score a group of replies with a reward function. No critic. | agentic source | E |
| Distillation | Train a smaller model from a larger one. | `build-efficient-ai-agents-through-model-distillation-with-nvidias-data-flywheel-blueprint.txt` | F |
| Instruction tuning | Post-training that teaches a model to follow instructions. | `ai-training.txt` | A, B |
| LLMs | Large models. Qwen 1.5B and 0.5B stand in. | none for a size limit | all parts |
| Small language models | A small model can serve narrow subtasks. | `how-small-language-models-are-key-to-scalable-agentic-ai.txt` | F (0.5B against 1.5B) |
| Reasoning models | Models built to think before they answer. Not run here. | `an-easy-introduction-to-llm-reasoning-ai-agents-and-test-time-scaling.txt` | A (room to think, in a model that is not one) |
| Base models | The model before instruction tuning. | none defines it | A |
| Instruction-tuned models | The model after instruction tuning. | none defines it | A |
| Open and proprietary models | Open weights can be trained by anyone who has them. Not measured. | `open-models-data-ai.txt` (open models) | none |
| Dense models | Not run. No MoE model is trained here. | none | none |
| Mixture-of-experts models | Named in the sources as a model type. Not run. | `ai-training.txt` | none |
| Foundation versus task-specific | A foundation model is trained on raw data. A task-specific model is tuned on a task dataset. | `ai-decoded-foundation-models.txt`, `ai-training.txt` | G (gain against forgetting) |

In [ ]:
# 20. Final cell: status of every part, files saved, stray processes stopped, GPU memory in use. No server runs in this notebook.
subprocess.run("pkill -f 'align/stages/' || true", shell=True)
print(pd.DataFrame([dict(part=k, status=v[0], seconds=round(v[1])) for k, v in PARTS.items()]).to_string(index=False))
print(pd.DataFrame([dict(stage=k, status=v[0], seconds=v[1]) for k, v in STAGE.items()]).to_string(index=False))
import textwrap
print("saved in", OUT, "(csv and png):"); print(textwrap.fill(", ".join(sorted(p.name for p in OUT.glob("*.csv")) + sorted(p.name for p in OUT.glob("*.png"))), 150)); print("raw replies per stage in", W / "res")
try: print("GPU memory in use (MiB):", subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader"], capture_output=True, text=True, timeout=30).stdout.split())
except Exception as e: print("nvidia-smi not available:", type(e).__name__)
if not str(M).startswith("/kaggle/temp"): shutil.rmtree(M, ignore_errors=True)   # keep the output folder small
lap("done")